In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import os
import hashlib
from PIL import Image
import numpy as np
import random
from torch.utils.data import Subset

RuntimeError: Only a single TORCH_LIBRARY can be used to register the namespace prims; please put all of your definitions in a single TORCH_LIBRARY block.  If you were trying to specify implementations, consider using TORCH_LIBRARY_IMPL (which can be duplicated).  If you really intended to define operators for a single namespace in a distributed way, you can use TORCH_LIBRARY_FRAGMENT to explicitly indicate this.  Previous registration of TORCH_LIBRARY was registered at /dev/null:241; latest registration was registered at /dev/null:241

### Hardware and Project Configs

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# parallel processing fuer schnelleres trainieren des models
NUM_WORKERS = 0

# hyperparameters fuer lokales setup 
BATCH_SIZE = 64
IMG_SIZE = 100 
EPOCHS = 30
LEARNING_RATE = 0.001
NUM_CLASSES = 33

# pfade der bilder
TRAIN_DIR = "Fruits/train/train"   
TEST_DIR = "Fruits/test/test"

print(f"configs fuer diesen run:\nworkers: {NUM_WORKERS}  | batch size: {BATCH_SIZE} | on device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

### Data Engineering: Integrity check

In [ ]:
# diese listen speichern daten fuer die plots
fruit_names = []
fruit_counts = []


all_items = os.listdir(TRAIN_DIR)                   # extrahiere namen der folder

for item_name in all_items:                         # loop through every item found                      

    item_path = os.path.join(TRAIN_DIR, item_name)  # create the full path to the item
    

    if os.path.isdir(item_path):                    # erst checken ob es sich um ein directory handelt
        files_inside = os.listdir(item_path)        # alle files auflisten
        number_of_images = len(files_inside)        # count how many files are in there
        
        fruit_names.append(item_name)               # append namen in die liste
        fruit_counts.append(number_of_images)       # append count in die liste


plt.figure(figsize=(15, 6))                         # erstellt die figure fuer den plot mit (width=15, height=6)
plt.bar(fruit_names, fruit_counts, color='skyblue') 
plt.title("how many images does each fruit have?")  
plt.ylabel("number of images")                     
plt.xticks(rotation=90)                             
plt.show()



plt.figure(figsize=(10, 10))                        # create a new figure for the images
for i in range(9):
    random_fruit = random.choice(fruit_names)       # pick a random fruit name from our list
    fruit_folder = os.path.join(TRAIN_DIR, random_fruit)
    files_in_fruit_folder = os.listdir(fruit_folder)
    random_file = random.choice(files_in_fruit_folder)
    image_location = os.path.join(fruit_folder, random_file)
    
    img = Image.open(image_location)
    
   
    plt.subplot(3, 3, i + 1)                        # sublot fuer ein 3x3 feld
                                                    # i + 1 weil matplotlib erst mit 1 beginnt, nicht mit 0 (wegen der range() function)
    plt.imshow(img)
    plt.title(random_fruit)
    plt.axis('off')


plt.show()

### Data Pipeline and Augmentation

In [ ]:
# datenverarbeitungs- und augmentationsblock
train_transform = transforms.Compose([                                  # 1. positionale transformationen
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)), 
    transforms.RandomAffine(degrees=45, translate=(0.2, 0.2)), 
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5), 
    transforms.RandomRotation(20),

    transforms.ColorJitter(                                             # 2. licht und farbe
        brightness=0.4, 
        contrast=0.4, 
        saturation=0.4, 
        hue=0.05 
    ),
    transforms.RandomGrayscale(p=0.1),                                  # 3. 10% der farbe auf greyscale um konturen zu lernen      

    transforms.ToTensor(),                                              # 4. Wandelt Bilder in einen Tensor (mehrdimensionale Matrix) um
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))              #    und normalisiert diese auf -1 bis 1 für Stabileres Training
])


val_transform = transforms.Compose([                                    # validation ohne transformationen composen
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [ ]:
raw_dataset = datasets.ImageFolder(TRAIN_DIR)   # wir laden den ordner einmal komplett, nur um zu zählen, wie viele bilder da sind

count_train = int(0.8 * len(raw_dataset))       # berechne die anzahl: 80% für training, der rest für validierung
count_val = len(raw_dataset) - count_train



train_ids, val_ids = random_split(range(len(raw_dataset)), [count_train, count_val])    # wir erstellen zwei listen mit zufälligen id-nummern (z.b. 1, 5, 99... und 2, 3, 10...)

dataset_with_noise = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)         # transformierte bilder 
dataset_clean = datasets.ImageFolder(TRAIN_DIR, transform=val_transform)                # cleane bilder

final_train_subset = Subset(dataset_with_noise, train_ids)
final_val_subset = Subset(dataset_clean, val_ids)


train_loader = DataLoader(final_train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory = True) # der loader fuettert das modell haeppchenweise (batch_size) mit bildern
val_loader = DataLoader(final_val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory = True)


class_names = raw_dataset.classes                                                       # klassennamen speichern (apfel, banane, etc.)

### Model Architecture

In [ ]:
class FruitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        # 1. feature extraction (sehen)
        self.features = nn.Sequential(
            # block 1: kanten finden
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # eingang: 3 farben -> 32 merkmale
            nn.BatchNorm2d(32),                          # stabilisiert das lernen
            nn.ReLU(),                                   # aktiviert neuronen
            nn.MaxPool2d(2),                             # bild verkleinern (100 -> 50px)

            # block 2: formen finden
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # 32 -> 64 merkmale
            nn.BatchNorm2d(64),                          # stabilisierung
            nn.ReLU(),                                   # aktivierung
            nn.MaxPool2d(2),                             # verkleinern (50 -> 25px)

            # block 3: texturen finden
            nn.Conv2d(64, 128, kernel_size=3, padding=1),# 64 -> 128 merkmale
            nn.BatchNorm2d(128),                         # stabilisierung
            nn.ReLU(),                                   # aktivierung
            nn.MaxPool2d(2)                              # verkleinern (25 -> 12px)
        )
        
        # 2. entscheiden (classifier)
        # wir haben jetzt 128 merkmale * 12 * 12 pixel größe
        self.flatten_dim = 128 * 12 * 12
        
        self.classifier = nn.Sequential(
            nn.Flatten(),                     # alles in eine lange reihe schreiben
            nn.Linear(self.flatten_dim, 512), # erste denkschicht
            nn.ReLU(),                        # aktivierung
            nn.Dropout(0.5),                  # 50% vergessen (gegen overfitting!)
            nn.Linear(512, NUM_CLASSES)       # ausgabe: unsere früchte
        )

    def forward(self, x):
        x = self.features(x)   # erst gucken
        x = self.classifier(x) # dann entscheiden
        return x

# modell erstellen und auf die grafikkarte schieben
model = FruitCNN().to(DEVICE)

### Training Loop

In [ ]:
# ==========================================
# 5. training (mit scheduler & checkpoint)
# ==========================================
import torch.optim as optim
import matplotlib.pyplot as plt
import copy  # brauchen wir zum kopieren des besten modells

# werkzeuge
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# "brems-assistent" (scheduler)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

# listen für die kurven
train_losses = []
val_losses = []

# variablen für den rekord-halter
best_val_loss = float('inf')                          # am anfang ist unendlich der "beste" wert
best_model_wts = copy.deepcopy(model.state_dict()) # wir merken uns den start-zustand



for epoch in range(EPOCHS):
    
    #1. lernen (training)
    model.train()                                                   # lern-modus an
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)       # ab auf die grafikkarte
        
        optimizer.zero_grad()                                       # gedaechtnis löschen
        outputs = model(images)                                     # raten
        loss = criterion(outputs, labels)                           # fehler berechnen
        loss.backward()                                             # lernen
        optimizer.step()                                            # verbessern
        
        running_loss += loss.item()                                 # fehler merken

    # 2. pruefen (validierung)
    model.eval()                                                    # lern-modus aus (nur pruefen)
    val_running_loss = 0.0 
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(images)                                 # raten
            
            # fehler auch bei der pruefung berechnen
            loss = criterion(outputs, labels)
            val_running_loss += loss.item()
            
            # genauigkeit berechnen
            _, preds = torch.max(outputs, 1)                        # beste antwort wählen
            correct += (preds == labels).sum().item()               # richtige zählen
            total += labels.size(0)                                 # anzahl bilder zählen

    # --- statistik ---
    avg_train_loss = running_loss / len(train_loader)               # durchschnitts-fehler train
    avg_val_loss = val_running_loss / len(val_loader)               # durchschnitts-fehler val
    
    acc = 100 * correct / total                                     # genauigkeit in %
    
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    
    # NEU: scheduler fragen (ohne verbose)
    scheduler.step(avg_val_loss)
    
    # aktuelle lernrate abfragen (damit wir sehen, ob gebremst wurde)
    current_lr = optimizer.param_groups[0]['lr']
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_wts = copy.deepcopy(model.state_dict())          # bestes wissen sichern

    # hier zeigen wir jetzt auch die 'LR' (lernrate) an
    print(f"Runde {epoch+1}: Train={avg_train_loss:.4f} | Val={avg_val_loss:.4f} | Acc={acc:.1f}% | LR={current_lr:.6f}")


# am ende laden wir den besten zustand zurück
model.load_state_dict(best_model_wts)



In [ ]:
# grafik zeichnen
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='training loss', color='blue')
plt.plot(val_losses, label='validation loss', color='orange', linestyle='--')
plt.title("lern verlauf")
plt.xlabel("runden")
plt.ylabel("fehler")
plt.legend(); plt.grid(True); plt.show()

In [ ]:
# ==========================================
# 7. der internet-test (neue bilder testen)
# ==========================================
import os
from PIL import Image # python image library
import torch.nn.functional as F

# wo liegen die bilder?
# "." bedeutet "aktueller ordner", also da wo dein notebook liegt
TEST_DIR = "./downloads" 
IMG_SIZE = 100  # wichtig: muss exakt die größe aus dem training sein!

# wir müssen die bilder genau so behandeln wie im training (clean_transforms)
custom_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# alle bilder aus dem ordner laden
image_files = [f for f in os.listdir(TEST_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))]

if len(image_files) == 0:
    print(f"❌ Keine Bilder in '{TEST_DIR}' gefunden! Bitte Pfad prüfen.")
else:
    print(f"🔎 Teste {len(image_files)} externe Bilder...")

    plt.figure(figsize=(15, 10))
    model.eval() # prüfungs-modus

    for i, img_name in enumerate(image_files):
        # 1. bild laden
        img_path = os.path.join(TEST_DIR, img_name)
        image = Image.open(img_path).convert('RGB') # wichtig: falls png transparenz hat, entfernen
        
        # 2. bild vorbereiten (transformieren & batch-dimension hinzufügen)
        # aus (3, 100, 100) wird (1, 3, 100, 100)
        input_tensor = custom_transform(image).unsqueeze(0).to(DEVICE)
        
        # 3. vorhersage
        with torch.no_grad():
            outputs = model(input_tensor)
            probs = F.softmax(outputs, dim=1) # in prozent umwandeln
            conf, pred_idx = torch.max(probs, 1)
        
        # 4. ergebnis anzeigen
        predicted_class = class_names[pred_idx.item()]
        confidence = conf.item() * 100
        
        # bild für anzeige vorbereiten (denormalisieren)
        ax = plt.subplot(4, 4, i+1)
        
        # tensor zurückwandeln für matplotlib
        img_display = input_tensor.squeeze(0).cpu().permute(1, 2, 0).numpy()
        img_display = img_display * 0.5 + 0.5 # farben korrigieren
        img_display = np.clip(img_display, 0, 1)
        
        plt.imshow(img_display)
        plt.axis('off')
        
        # titel formatieren
        # farbe: grün bei hoher sicherheit (>80%), gelb bei unsicherheit, rot bei rateversuchen
        title_color = 'green' if confidence > 80 else 'orange'
        if confidence < 40: title_color = 'red'
            
        plt.title(f"{predicted_class}\n({confidence:.1f}%)", 
                  color=title_color, fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()